# R을 이용한 통계 분석

R 내장 데이터셋을 활용한 가설 검정, 회귀 분석 및 신뢰 구간 산출.

데이터 다운로드나 패키지 설치가 필요 없습니다 — 기본(Base) R만 사용합니다.

## 1. 기술 통계

In [ ]:
data(mtcars)
cat("Dataset: mtcars (", nrow(mtcars), "cars, ", ncol(mtcars), "variables)\n\n")
summary(mtcars[, c("mpg", "hp", "wt", "disp")])

## 2. 독립 2표본 t-검정

수동변속기 차량이 자동변속기 차량보다 연비(MPG)가 더 좋을까요?

In [ ]:
auto <- mtcars$mpg[mtcars$am == 0]
manual <- mtcars$mpg[mtcars$am == 1]

cat("Automatic:", round(mean(auto), 1), "mpg (n =", length(auto), ")\n")
cat("Manual:   ", round(mean(manual), 1), "mpg (n =", length(manual), ")\n\n")

t_result <- t.test(manual, auto, alternative = "greater")
print(t_result)

cat("\nConclusion:",
    ifelse(t_result$p.value < 0.05,
           "Reject H0 — manual cars have significantly higher MPG",
           "Fail to reject H0"))

## 3. 카이제곱 검정

실린더 수와 변속기 유형은 서로 독립일까요?

In [ ]:
tab <- table(Cylinders = mtcars$cyl, Transmission = mtcars$am)
colnames(tab) <- c("Automatic", "Manual")
print(tab)
cat("\n")
chisq.test(tab)

## 4. 다중 선형 회귀 분석

In [ ]:
model <- lm(mpg ~ wt + hp + am, data = mtcars)
summary(model)

## 5. 회귀 진단

In [ ]:
par(mfrow = c(2, 2))
plot(model)

## 6. 신뢰 구간

In [ ]:
ci <- confint(model, level = 0.95)
cat("95% Confidence Intervals:\n")
print(round(ci, 4))

In [ ]:
coefs <- coef(model)[-1]
ci_vals <- ci[-1, ]
n <- length(coefs)

par(mfrow = c(1, 1), mar = c(5, 8, 4, 2))
plot(coefs, 1:n, xlim = range(ci_vals),
     pch = 19, col = "#58a6ff", cex = 1.5,
     yaxt = "n", xlab = "Estimate", ylab = "",
     main = "95% Confidence Intervals for Coefficients")
axis(2, at = 1:n, labels = names(coefs), las = 1)
segments(ci_vals[, 1], 1:n, ci_vals[, 2], 1:n,
         lwd = 3, col = "#58a6ff")
abline(v = 0, lty = 2, col = "#f85149", lwd = 1.5)

## 7. 일원 배치 분산 분석 (One-Way ANOVA)

실린더 수에 따라 연비(MPG)에 유의미한 차이가 있을까요?

In [ ]:
anova_model <- aov(mpg ~ factor(cyl), data = mtcars)
summary(anova_model)
cat("\nTukey HSD Post-hoc Comparisons:\n")
TukeyHSD(anova_model)

In [ ]:
boxplot(mpg ~ cyl, data = mtcars,
        main = "MPG by Cylinder Count",
        xlab = "Cylinders", ylab = "Miles per Gallon",
        col = c("#58a6ff", "#a371f7", "#f85149"))

## 요약

- **웰치의 t-검정**: 보정되지 않은 단측 비교에서 수동변속기 차량의 평균 연비(MPG)가 더 높게 나타납니다.
- **카이제곱 검정**: 분할표는 연관성을 시사하지만, 작은 기댓값으로 인해 근사치 경고가 발생하므로 이 결과를 신중하게 해석해야 합니다.
- **회귀 분석**: 변수 조정 후 차량 무게(weight)와 마력(horsepower)은 유의미한 음의 예측 변수이며, 이 모델에서 변속기 유형은 유의미하지 않습니다.
- **분산 분석 (ANOVA)**: 4기통, 6기통, 8기통 그룹 간에 연비 차이가 유의미하게 존재하며, 튜키(Tukey) 사후 검정 결과로 쌍별 차이를 확인할 수 있습니다.